# 这里是 -> **[本篇笔记博客](https://blog.algieba12.cn/llm04-rag-llamaindex-chromadb/)**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [14]:
!pip install -U transformers peft accelerate bitsandbytes sentence-transformers

# 2. 安装 LlamaIndex 核心及相关插件
!pip install llama-index-core llama-index-llms-huggingface llama-index-embeddings-huggingface

# 3. 安装 ChromaDB 向量库支持
!pip install llama-index-vector-stores-chroma chromadb

  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached huggingface_hub-1.3.4-py3-none-any.whl.metadata (13 kB)
Using cached transformers-5.0.0-py3-none-any.whl (10.1 MB)
Using cached huggingface_hub-1.3.4-py3-none-any.whl (536 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-huggingface 0.6.1 requires transformers[torch]<5,>=4.37.0, but you have transformers 5.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic

In [15]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_models"

In [16]:

# 创建数据目录
os.makedirs("/kaggle/working/data", exist_ok=True)

# 文档 1: 核心 API 定义
api_doc = """
[机密] DeepStar 核心交易接口 v2.0
1. 创建订单 API: POST /api/v2/order/create
   - 必填参数: 'user_id' (String), 'amount' (Decimal), 'token' (X-Auth-Token)
   - 特殊逻辑: 如果 amount > 10000, 必须额外传递 'audit_code' (审计码)。
   - 频率限制: 单用户每秒最多 5 次调用。
2. 查询余额 API: GET /api/v2/balance
   - 缓存策略: 默认缓存 5 秒。传递 'no-cache=true' 可强制刷新。
"""

# 文档 2: 错误码字典
error_doc = """
[机密] DeepStar 全局错误码字典
- E1001: 签名验证失败。请检查 X-Auth-Token 是否过期。
- E2009: 余额不足。注意：冻结金额不计入可用余额。
- E5003: 审计风控拦截。大额交易未通过自动审计，请联系人工客服。
"""

with open("/kaggle/working/data/api_specs.txt", "w") as f:
    f.write(api_doc)
with open("/kaggle/working/data/error_codes.txt", "w") as f:
    f.write(error_doc)

print("[Success] 企业文档库已就绪！")

[Success] 企业文档库已就绪！


In [17]:
import torch
from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. 设置 Embedding (眼睛)
# 使用 BGE-Small，显存占用极低，检索中文效果极佳
print("正在加载 Embedding 模型...")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-zh-v1.5"
)

# 2. 设置 LLM (大脑)
# 使用 Qwen2.5-7B-Instruct
print("正在加载 LLM 模型...")
Settings.llm = HuggingFaceLLM(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    tokenizer_name="Qwen/Qwen2.5-7B-Instruct",
    context_window=30000,
    max_new_tokens=512,
    generate_kwargs={"temperature": 0.1, "do_sample": True}, # 技术文档要求严谨，温度调低
    device_map="auto",
    model_kwargs={"torch_dtype": torch.float16, "trust_remote_code": True}
)
print("[Success] 模型加载完毕！")

正在加载 Embedding 模型...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[Success] 模型加载完毕！


In [18]:
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

# 定义持久化路径
CHROMA_DB_PATH = "/kaggle/working/chroma_db"
COLLECTION_NAME = "deepstar_docs"

# 1. 初始化 Chroma 客户端 (PersistentClient 实现了写硬盘功能)
db_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# 2. 创建或获取集合 (Collection)
chroma_collection = db_client.get_or_create_collection(COLLECTION_NAME)

# 3. 将 Chroma 对接给 LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. 智能加载逻辑 (幂等性设计)
if chroma_collection.count() == 0:
    print("[Info] 数据库为空，开始初始化...")
    # 读取 data 目录下的所有文件
    documents = SimpleDirectoryReader("./data").load_data()
    # 建立索引并自动存入 Chroma (Ingestion)
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context
    )
    print("[Success] 数据写入完成！")
else:
    print(f"[Info] 发现 {chroma_collection.count()} 条存量数据，直接加载...")
    # 直接从 Vector Store 加载，无需重新计算 Embedding
    index = VectorStoreIndex.from_vector_store(
        vector_store, storage_context=storage_context
    )
    print("[Success] 索引加载完成！")

[Info] 数据库为空，开始初始化...
[Success] 数据写入完成！


In [19]:
# 创建查询引擎
query_engine = index.as_query_engine(similarity_top_k=3)

# 实习生的提问
questions = [
    "创建订单时，如果你只有 100 块钱，能传 amount=20000 吗？为什么？",
    "我收到了 E5003 错误，这是什么意思？该怎么办？"
]

print("======== 开始 RAG 问答测试 ========")

for q in questions:
    print(f"\n[Question] {q}")
    response = query_engine.query(q)
    print(f"[Answer]\n{str(response)}")
    
    # 打印引用源 (Debug 必备，看看它参考了哪个文件)
    source_file = response.source_nodes[0].metadata.get('file_name')
    print(f"[Source]: {source_file}")

======== 开始 RAG 问答测试 ========

[Question] 创建订单时，如果你只有 100 块钱，能传 amount=20000 吗？为什么？
[Answer]
不能。因为创建订单的 amount 参数不能设置为 20000，原因如下：

1. **金额限制**：根据创建订单 API 的描述，amount 参数必须是一个 Decimal 类型的值，并且在没有额外传递 audit_code 的情况下，amount 不能超过 10000。因此，即使你有 100 块钱，也不能将 amount 设置为 20000。

2. **特殊逻辑**：如果 amount 大于 10000，则需要额外传递 audit_code。在这种情况下，amount 不能直接设置为 20000，因为它超过了允许的最大值。

综上所述，无论是从金额限制还是特殊逻辑的角度来看，amount 都不能设置为 20000。此外，即使你只有 100 块钱，也无法满足这个条件，因为 100 远远小于 10000，更不用说 20000 了。因此，正确的做法是确保 amount 的值在合理范围内，并且不超过 10000（如果有审计需求）。同时，还需要确保有足够的余额来完成交易，否则会收到 E2009 错误（余额不足）。 ---------------------
[Source]: api_specs.txt

[Question] 我收到了 E5003 错误，这是什么意思？该怎么办？
[Answer]
你收到了 E5003 错误，这意味着在进行大额交易时未通过自动审计。具体来说，你的交易金额超过了系统设定的阈值，需要进行额外的人工审核。

建议你采取以下步骤：
1. 联系 DeepStar 的人工客服，说明你的交易情况。
2. 等待人工审核的结果，并根据客服的指导进行后续操作。 在此期间，你可以检查是否有其他可能影响审核的因素，如账户状态、交易历史等。 保持与客服的良好沟通，以便尽快解决问题。 

如果未来有类似的大额交易需求，可以提前与客服沟通，了解具体的审核流程和所需材料，以减少等待时间。同时，确保账户信息准确无误，避免因信息错误导致审核延迟。 
以上是针对 E5003 错误的一些建议，希望对你有所帮助。如果有更多问题，欢迎随时咨询。祝您使用愉快！
E5003: 审计风控拦截。大额交易

In [20]:
# 偷看数据库里的前 2 条记录
data = chroma_collection.peek(limit=2)

print("\n[Debug] 数据库抽查:")
for i, doc in enumerate(data['documents']):
    print(f"--- 片段 {i} ---")
    print(f"内容: {doc[:50]}...") # 只打印前50个字
    print(f"来源: {data['metadatas'][i]}")


[Debug] 数据库抽查:
--- 片段 0 ---
内容: [机密] DeepStar 核心交易接口 v2.0
1. 创建订单 API: POST /api/v...
来源: {'_node_type': 'TextNode', 'last_modified_date': '2026-01-27', 'file_type': 'text/plain', 'creation_date': '2026-01-27', 'ref_doc_id': 'de4c5543-c20a-41f4-b26c-6e345cac3dcf', 'file_size': 440, 'doc_id': 'de4c5543-c20a-41f4-b26c-6e345cac3dcf', 'document_id': 'de4c5543-c20a-41f4-b26c-6e345cac3dcf', '_node_content': '{"id_": "effe3819-7968-4c57-9122-7dce5572b442", "embedding": null, "metadata": {"file_path": "/kaggle/working/data/api_specs.txt", "file_name": "api_specs.txt", "file_type": "text/plain", "file_size": 440, "creation_date": "2026-01-27", "last_modified_date": "2026-01-27"}, "excluded_embed_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "excluded_llm_metadata_keys": ["file_name", "file_type", "file_size", "creation_date", "last_modified_date", "last_accessed_date"], "relationships": {"1": {"node_id": "de4c5543-c20a-41f